# One-vs-Rest Classification Analysis

---
## 1. Setup & Data Loading

In [1]:
# =============================================================================
# LIBRARY IMPORTS
# =============================================================================

# Core Libraries
import numpy as np
import pandas as pd
import json
from pathlib import Path
import warnings
from collections import Counter

# Machine Learning Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, brier_score_loss, cohen_kappa_score, confusion_matrix
)

# Statistical Analysis
from scipy import stats
from scipy.stats import (
    wilcoxon, ttest_rel, mannwhitneyu, kruskal, 
    fisher_exact, chi2_contingency, binom
)
from statsmodels.stats.multitest import multipletests

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Configuration
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

# Color Palette
COLORS = {
    'ovr': '#e6550d',        # Orange
    'multinomial': '#2b8cbe', # Blue
    'rest': '#31a354',        # Green
    'task': '#de2d26',        # Red
    'positive': '#b2182b',
    'negative': '#2166ac',
    'neutral': '#f7f7f7'
}

print("✓ Libraries imported successfully")
print(f"  NumPy: {np.__version__}")
print(f"  Pandas: {pd.__version__}")

✓ Libraries imported successfully
  NumPy: 2.3.4
  Pandas: 2.3.3


In [2]:
# =============================================================================
# PATH CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path('/home/sjoon/projects/brain_connectivity_classifier')
RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'

# Full Model Paths (232 regions)
PATHS = {
    'full_ovr': RESULTS_DIR / 'full_connectivity_analysis' / 'one_vs_rest',
    'full_ovr_task': RESULTS_DIR / 'full_connectivity_analysis' / 'task_testing_one_vs_rest',
    # Left Hemisphere
    'left_ovr': RESULTS_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'one_vs_rest',
    'left_ovr_task': RESULTS_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'task_testing_one_vs_rest',
    # Right Hemisphere
    'right_ovr': RESULTS_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'one_vs_rest',
    'right_ovr_task': RESULTS_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'task_testing_one_vs_rest',
}

# Verify paths
print("Path Verification:")
for name, path in PATHS.items():
    status = "✓" if path.exists() else "✗"
    print(f"  {status} {name}")

Path Verification:
  ✓ full_ovr
  ✓ full_ovr_task
  ✗ left_ovr
  ✗ left_ovr_task
  ✗ right_ovr
  ✗ right_ovr_task


In [3]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def load_json(fp):
    """Load JSON file."""
    with open(fp, 'r') as f:
        return json.load(f)

def load_npy(fp):
    """Load numpy file."""
    return np.load(fp, allow_pickle=True)

def load_csv(fp):
    """Load CSV file."""
    return pd.read_csv(fp)

def safe_divide(num, denom, default=0):
    """Safe division with zero handling."""
    return num / denom if denom > 0 else default

class DataLoader:
    """Structured data loader for model results."""
    
    def __init__(self, base_path, is_task=False):
        self.base_path = Path(base_path)
        self.is_task = is_task
        
    def load_cv(self):
        """Load cross-validation results."""
        return {
            'predictions': load_npy(self.base_path / 'cv_predictions.npy'),
            'probabilities': load_npy(self.base_path / 'cv_probabilities.npy'),
            'true_labels': load_npy(self.base_path / 'cv_true_labels.npy'),
            'confusion_matrix': load_npy(self.base_path / 'confusion_matrix.npy'),
            'metrics': load_json(self.base_path / 'overall_metrics.json')
        }
    
    def load_task(self):
        """Load task testing results."""
        return {
            'predictions': load_npy(self.base_path / 'task_predictions.npy'),
            'probabilities': load_npy(self.base_path / 'task_probabilities.npy'),
            'true_labels': load_npy(self.base_path / 'task_true_labels.npy'),
            'confusion_matrix': load_npy(self.base_path / 'task_confusion_matrix.npy'),
            'summary': load_json(self.base_path / 'task_testing_summary.json')
        }

print("✓ Helper functions defined")

✓ Helper functions defined


In [4]:
# =============================================================================
# DATA LOADING
# =============================================================================

print("="*80)
print("LOADING ALL MODEL DATA")
print("="*80)

# Full Model (232 regions)
print("\n[Full Model - 232 regions]")
full_ovr_cv = DataLoader(PATHS['full_ovr']).load_cv()
full_ovr_task = DataLoader(PATHS['full_ovr_task']).load_task()
print(f"  OvR CV: {len(full_ovr_cv['predictions']):,} samples")
print(f"  OvR Task: {len(full_ovr_task['predictions']):,} samples")

# # Left Hemisphere (116 regions)
# print("\n[Left Hemisphere - 116 regions]")
# left_ovr_cv = DataLoader(PATHS['left_ovr']).load_cv()
# left_ovr_task = DataLoader(PATHS['left_ovr_task']).load_task()
# print(f"  OvR CV: {len(left_ovr_cv['predictions']):,} samples")

# # Right Hemisphere (116 regions)
# print("\n[Right Hemisphere - 116 regions]")
# right_ovr_cv = DataLoader(PATHS['right_ovr']).load_cv()
# right_ovr_task = DataLoader(PATHS['right_ovr_task']).load_task()
# print(f"  OvR CV: {len(right_ovr_cv['predictions']):,} samples")

# Region Information
region_info = load_csv('/home/sjoon/projects/brain_connectivity_classifier/data/region_info.csv')
print(f"\n✓ Loaded region info: {len(region_info)} regions")

LOADING ALL MODEL DATA

[Full Model - 232 regions]
  OvR CV: 51,968 samples
  OvR Task: 46,400 samples

✓ Loaded region info: 232 regions


### Recall Vs Precision (232 Regions)

In [9]:
# =============================================================================
# FIXED & IMPROVED: PER-REGION DETAILED ERROR PROFILE ANALYSIS
# =============================================================================

import numpy as np
import pandas as pd

# 1. Define Yeo-17 networks for accurate cortical/subcortical tagging
yeo17_networks = [
    'VisCent', 'VisPeri', 'SomMotA', 'SomMotB',
    'DorsAttnA', 'DorsAttnB', 'SalVentAttnA', 'SalVentAttnB',
    'LimbicA', 'LimbicB',  # Note: order doesn't matter
    'ContA', 'ContB', 'ContC',
    'DefaultA', 'DefaultB', 'DefaultC',
    'TempPar'
]

# Create Atlas column (robust, no dependency on previous 'Type')
region_info['Atlas'] = np.where(
    region_info['network'].isin(yeo17_networks),
    'Schaefer17',
    'Tian'
)

# 2. Compute Recall (class-wise accuracy) and Precision
def get_recall(y_t, y_p): 
    return pd.Series(y_p == y_t).groupby(y_t).mean() * 100

def get_precision(y_t, y_p):
    return pd.Series(y_t == y_p).groupby(y_p).mean() * 100

# Recall (aligned to region_info index)
region_info['Rest_Recall'] = get_recall(full_ovr_cv['true_labels'], full_ovr_cv['predictions'])
region_info['Task_Recall'] = get_recall(full_ovr_task['true_labels'], full_ovr_task['predictions'])
region_info['Recall_Gap'] = region_info['Rest_Recall'] - region_info['Task_Recall']

# Precision (reindex handles regions never predicted)
region_info['Rest_Precision'] = get_precision(
    full_ovr_cv['true_labels'], full_ovr_cv['predictions']
).reindex(region_info.index, fill_value=np.nan)

region_info['Task_Precision'] = get_precision(
    full_ovr_task['true_labels'], full_ovr_task['predictions']
).reindex(region_info.index, fill_value=np.nan)

region_info['Precision_Gap'] = region_info['Rest_Precision'] - region_info['Task_Precision']

# Relative drops
region_info['Recall_Rel_Drop_%'] = (
    region_info['Recall_Gap'] / region_info['Rest_Recall'].replace(0, np.nan) * 100
)
region_info['Precision_Rel_Drop_%'] = (
    region_info['Precision_Gap'] / region_info['Rest_Precision'].replace(0, np.nan) * 100
)

# 3. Improved printing function (now uses 'Atlas' instead of missing 'Type')
def print_per_region_drop(
    df,
    sort_by='Recall_Gap',
    title_suffix='',
    top_n=30,
    width=180
):
    sorted_df = df.sort_values(sort_by, ascending=False).copy()
    if top_n is not None:
        sorted_df = sorted_df.head(top_n)
        top_str = f"(Top {top_n}) "
    else:
        top_str = ""
    
    print(f"\n{'='*width}")
    print(f"PER-REGION ERROR PROFILE {top_str}{title_suffix}")
    print(f"Sorted by descending {sort_by}")
    print("Recall drop → increased false negatives (harder to detect in task)")
    print("Precision drop → increased false positives (less specific in task)")
    print(f"{'='*width}")
    
    header = (
        f"{'Region':<40} {'Network':<25} {'Atlas':<10} │ "
        f"{'Rest Rec':>10} {'Task Rec':>10} {'Abs Drop':>11} {'Rel Drop':>11} │ "
        f"{'Rest Prec':>11} {'Task Prec':>11} {'Abs Drop':>11} {'Rel Drop':>11}"
    )
    print(header)
    print('-' * width)
    
    for _, row in sorted_df.iterrows():
        rest_rec = f"{row['Rest_Recall']:10.2f}%"
        task_rec = f"{row['Task_Recall']:10.2f}%"
        gap_rec = f"{row['Recall_Gap']:10.2f}pt" if pd.notna(row['Recall_Gap']) else "    -     "
        rel_rec = f"{row['Recall_Rel_Drop_%']:10.1f}%" if pd.notna(row['Recall_Rel_Drop_%']) else "    -     "
        
        rest_prec = f"{row['Rest_Precision']:11.2f}%" if pd.notna(row['Rest_Precision']) else "     -     "
        task_prec = f"{row['Task_Precision']:11.2f}%" if pd.notna(row['Task_Precision']) else "     -     "
        gap_prec = f"{row['Precision_Gap']:10.2f}pt" if pd.notna(row['Precision_Gap']) else "    -     "
        rel_prec = f"{row['Precision_Rel_Drop_%']:10.1f}%" if pd.notna(row['Precision_Rel_Drop_%']) else "    -     "
        
        print(
            f"{row['region_name']:<40} {row['network']:<25} {row['Atlas']:<10} │ "
            f"{rest_rec} {task_rec} {gap_rec} {rel_rec} │ "
            f"{rest_prec} {task_prec} {gap_prec} {rel_prec}"
        )
    
    return sorted_df

# 4. Execute the tables
print("\n" + "="*100)
print("DETAILED PER-REGION ERROR PROFILE ANALYSIS (TOP 30 WORST)")
print("="*100)

# Primary: largest absolute recall drops
_ = print_per_region_drop(
    region_info,
    sort_by='Recall_Gap',
    title_suffix="LARGEST ABSOLUTE RECALL DROP (MOST FN-INCREASE)",
    top_n=30
)

# Secondary: largest relative recall drops
_ = print_per_region_drop(
    region_info,
    sort_by='Recall_Rel_Drop_%',
    title_suffix="LARGEST RELATIVE RECALL DROP (PROPORTIONALLY HARDEST HIT)",
    top_n=30
)

# Tertiary: largest precision drops
_ = print_per_region_drop(
    region_info,
    sort_by='Precision_Gap',
    title_suffix="LARGEST ABSOLUTE PRECISION DROP (MOST FP-INCREASE)",
    top_n=30
)


DETAILED PER-REGION ERROR PROFILE ANALYSIS (TOP 30 WORST)

PER-REGION ERROR PROFILE (Top 30) LARGEST ABSOLUTE RECALL DROP (MOST FN-INCREASE)
Sorted by descending Recall_Gap
Recall drop → increased false negatives (harder to detect in task)
Precision drop → increased false positives (less specific in task)
Region                                   Network                   Atlas      │   Rest Rec   Task Rec    Abs Drop    Rel Drop │   Rest Prec   Task Prec    Abs Drop    Rel Drop
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
LH_DefaultB_Temp_3                       DefaultB                  Schaefer17 │      81.25%      61.50%      19.75pt       24.3% │       81.98%       87.86%      -5.88pt       -7.2%
LH_ContA_Cingm_1                         ContA                     Schaefer17 │      80.80%      62.50%      18.30pt       22.7% │       71.26%       60.

In [15]:
import plotly.express as px
import plotly.graph_objects as go

def plot_gap_scatter_readable(df):
    # 1. Create the main scatter plot with marginal distributions (box plots)
    fig = px.scatter(
        df,
        x='Recall_Gap',
        y='Precision_Gap',
        color='Atlas',
        symbol='Atlas',  # Added symbol for accessibility/clarity
        hover_name='region_name',
        # Add 'network' to hover data to spot patterns
        hover_data={
            'network': True, 
            'Recall_Gap': ':.2f', 
            'Precision_Gap': ':.2f', 
            'Atlas': False
        },
        color_discrete_map={'Schaefer17': '#1f77b4', 'Tian': '#ff7f0e'},
        title="<b>Performance Shift: Rest → Task</b><br><sup>Positive values indicate performance degradation (Gap > 0)</sup>",
        labels={
            'Recall_Gap': 'Recall Gap (Positive = More False Negatives)', 
            'Precision_Gap': 'Precision Gap (Positive = More False Positives)'
        },
        marginal_x="box", # Shows distribution of Recall Gaps
        marginal_y="box", # Shows distribution of Precision Gaps
        template="plotly_white" # Cleaner background for readability
    )

    # 2. Refine the markers
    fig.update_traces(marker=dict(size=10, opacity=0.7, line=dict(width=1, color='DarkSlateGrey')))

    # 3. Add Quadrant Lines (Zero lines)
    fig.add_vline(x=0, line_dash="solid", line_color="black", line_width=1)
    fig.add_hline(y=0, line_dash="solid", line_color="black", line_width=1)

    # 4. Dynamic Quadrant Annotations
    # We use x_ref="paper" to place text relative to the layout (0 to 1) 
    # rather than data coordinates. This prevents text from disappearing if data range changes.
    
    # Top Right (Both Worse)
    fig.add_annotation(
        xref="x domain", yref="y domain",
        x=0.98, y=0.98, showarrow=False,
        text="<b>Both Degrade</b><br>(FN & FP ↑)",
        font=dict(color="red", size=10), align="right",
        bgcolor="rgba(255,255,255,0.8)"
    )

    # Bottom Left (Both Better)
    fig.add_annotation(
        xref="x domain", yref="y domain",
        x=0.02, y=0.02, showarrow=False,
        text="<b>Both Improve</b><br>(Performance ↑)",
        font=dict(color="green", size=10), align="left",
        bgcolor="rgba(255,255,255,0.8)"
    )

    # Top Left (FN Improve, FP Worsen) - Tradeoff
    fig.add_annotation(
        xref="x domain", yref="y domain",
        x=0.02, y=0.98, showarrow=False,
        text="<b>Precision Drops</b><br>(More False Positives)",
        font=dict(color="gray", size=10), align="left"
    )

    # Bottom Right (FN Worsen, FP Improve) - Tradeoff
    fig.add_annotation(
        xref="x domain", yref="y domain",
        x=0.98, y=0.02, showarrow=False,
        text="<b>Recall Drops</b><br>(More False Negatives)",
        font=dict(color="gray", size=10), align="right"
    )

    # 5. Final Layout Polish
    fig.update_layout(
        height=700, 
        width=900, 
        legend_title="Brain Atlas",
        font=dict(family="Arial", size=12),
        legend=dict(
            yanchor="top", y=0.99,
            xanchor="left", x=0.01,
            bgcolor="rgba(255,255,255,0.9)"
        )
    )
    
    fig.show()

# Run it
plot_gap_scatter_readable(region_info)

### Recall By Network

In [33]:
# =============================================================================
# 3a. Interactive Box Plot: Recall Gap by Network (separate)
# =============================================================================
def plot_recall_gap_by_network_plotly(df):
    fig = px.box(
        df,
        x='network',
        y='Recall_Gap',
        color='Atlas',
        points='outliers',
        hover_name='region_name',
        hover_data={'network': True, 'Recall_Gap': ':.2f'},
        color_discrete_map={'Schaefer17': '#1f77b4', 'Tian': '#ff7f0e'},
        title='Recall Gap Distribution by Network<br>(positive = more FN / less detectable)',
        labels={'Recall_Gap': 'Gap (percentage points)', 'network': 'Network'}
    )
    
    # Dashed zero line
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.8)
    
    # Fixed Y-axis as requested
    fig.update_yaxes(range=[-20, 20])
    
    fig.update_layout(
        height=500,
        width=1200,
        xaxis=dict(tickangle=90, tickfont_size=10),
        legend_title="Atlas"
    )
    
    fig.show()

plot_recall_gap_by_network_plotly(region_info)


Recall
- Recall is TP/(TP+FN) -> Recall means out of all actual cases (Positive) how many it can find correctly. 
- Recall is important in medical cases where we can have more FP because it is more risky classify a patient FN. we can do another test to confirm the patient health. 

Recall Gap
- Recall Gap (Rest Recal  - Task Recal)
- Positive gap → recall drops in task → more false negatives (FN) → the region becomes harder to detect (less detectable/sensitive).
- Negative gap → recall improves in task

Task conditions make many regions harder to detect (↑FN Positive Gap), especially the subcortical regions 

Observations (subcortial)
- Very high positive medians and large outliers (e.g., Pallidum_post, Hippocampus_post, Putamen, Accumbens).
- Gaps often +10 to +20 pt or more → strong recall drops.
- These small/deep structures become much less detectable during task (likely because task alters their connectivity patterns or adds noise).


Observations (cortical)
- Medians mostly near 0 or slightly positive.
- Smaller spreads and fewer extreme outliers.
- Primary sensory/motor networks (VisCent, VisPeri, SomMot) show almost no change.
- Higher-order networks (ContA/B, Default, SalVentAttn) have modest positive medians → some detectability loss.

The biggest detectability problems in task are concentrated in subcortical regions (hippocampus, pallidum, basal ganglia). Cortical regions are more stable, especially early sensory ones.


### Precision By Network

In [ ]:
# =============================================================================
# 3b. Interactive Box Plot: Precision Gap by Network (separate)
# =============================================================================
def plot_precision_gap_by_network_plotly(df):
    fig = px.box(
        df,
        x='network',
        y='Precision_Gap',
        color='Atlas',
        points='outliers',
        hover_name='region_name',
        hover_data={'network': True, 'Precision_Gap': ':.2f'},
        color_discrete_map=atlas_colors,
        title='Precision Gap Distribution by Network<br>(positive = more FP / less specific)',
        labels={'Precision_Gap': 'Gap (percentage points)', 'network': 'Network'}
    )
    
    # Dashed zero line
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.8)
    
    # Fixed Y-axis as requested
    fig.update_yaxes(range=[-30, 30])
    
    fig.update_layout(
        height=500,
        width=1200,
        xaxis=dict(tickangle=90, tickfont_size=10),
        legend_title="Atlas"
    )
    
    fig.show()

plot_precision_gap_by_network_plotly(region_info)

Overall Takeaways

- Subcortical vulnerability: Tian networks (orange) consistently show the worst recall degradation and some of the biggest precision hits → small/deep structures lose both detectability and specificity in task.
- Cortical resilience (stable) in early areas: Visual, somatomotor networks barely change.
- Higher-order cortical sensitivity: Control, default, and attention networks show moderate recall drops and variable precision changes → task alters their functional segregation.

Different failure modes:
- Recall plot → mainly FN increase (missed detections), worst in subcortical.
- Precision plot → mainly FP increase (over-detections), more scattered but extreme in control/subcortical.


These patterns suggest task demands disrupt functional segregation most in subcortical and executive/control systems — classic finding in task-based fMRI connectivity studies.